# CALIBRAÇÃO DO GERADOR POR REGIÃO
Procura, com o `NatureSelector` (`genetic`, CMA-ES), a configuração do `SyntheticGenerator` que faz
os tiles sintéticos se parecerem com os tiles reais de cada região de Marlim — `calm`, `faulted` e
`dead`, separados em `../Marlim/Analysis.ipynb`. O resultado vai para `regions.json`, que é o que o
`../Generate.ipynb` usa para montar o dataset de treino.

O objetivo é a **similaridade**: uma porcentagem entre um lote de `N_IMAGES` tiles sintéticos e todos
os tiles reais da região, pelo coeficiente de energia normalizado sobre uma régua de 37 features
sísmicas, com dois grupos dedicados a falha. O **`README.md`** desta pasta explica a função inteira.

Escala medida (README, seção 7): **99%** é o teto — tiles reais contra a própria região —, **97%** é
um patch inteiro contra o resto da sua região, **58–79%** é uma região diferente, e as três
configurações calibradas à mão no `Generator2.ipynb` ficam entre **73% e 89%**.

O notebook não depende de nenhum arquivo de configuração: o melhor genoma conhecido de cada região é
`Generator.REFERENCE`, constante da classe, e a única coisa que ele lê de fora são os tiles reais.
`regions.json` é **saída**, não entrada — quando existe, o que está lá também disputa a escolha final.

Cada busca usa `memory=files/memory/<região>_<caixa>`: interrompida, a próxima chamada retoma de onde
parou; terminada, a próxima **estende** a campanha. A caixa entra no nome da pasta de propósito —
o estado do CMA-ES está em unidades reais, então retomar uma campanha com outra caixa partiria de uma
média fora dela. E como `export` compara o achado novo com o que já estava gravado, rodar de novo
nunca piora o `regions.json`.

In [1]:
import os, sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
import cv2
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime
from PIL import Image

sys.path.append('../../..')
from Synthetic.index import SyntheticGenerator
from Nature.index import NatureSelector
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [2]:
REAL_DIR     = '../Marlim/files'
PATCH_DIR    = '../../files/patches'
MEMORY_DIR   = 'files/memory'
REGIONS_PATH = 'regions.json'
REGIONS      = ('calm', 'faulted', 'dead')

TILE, SECTIONS = 128, (16, 48, 80, 112)   # tile e as seções inline/crossline medidas em cada tile
N_IMAGES       = 18                       # tiles por lote: ruído de ~1 ponto na nota (README, seção 7)
SEED           = 20260911                 # sementes da busca: a mesma configuração devolve sempre a mesma nota
CHECK_SEED     = SEED + 500               # sementes novas, para a escolha final e a nota honesta
CLIP           = 0.45                     # saturação simétrica, como no slab real
RESOLUTION     = 0.25                     # ε de cada feature, em IQR dela dentro da própria região
SEM_MIN        = 0.8                      # semblance abaixo disso marca o voxel como descontínuo
LINE_MIN       = 24                       # comprimento mínimo de um lineamento de descontinuidade, em px
LINE_DIP       = (35, 88)                 # faixa de mergulho aceita para lineamento tipo falha
NJOBS          = min(18, os.cpu_count())

POPULATION, GENERATIONS, PATIENCE = 12, 100, 10            # uma chamada: 84 avaliações, ~60 min por região
SPREAD = {'calm': 0.20, 'faulted': 0.15, 'dead': 0.20}     # meia-largura da caixa, em fração da faixa global

N_IMAGES, POPULATION * GENERATIONS, SPREAD

(18, 1200, {'calm': 0.2, 'faulted': 0.15, 'dead': 0.2})

# RÉGUA
As 37 features de uma seção 2D, calculadas pelo mesmo código no real e no sintético, em 6 grupos:
`amplitude`, `espectro`, `estrutura`, `continuidade`, `falhas` e `rotulo`. Os dois últimos são os que
enxergam falha — o primeiro pela **semblance com direção de mergulho** (coerência no sentido de
Marfurt et al., 1998), sem usar rótulo, e o segundo confrontando rótulo e imagem, com destaque para a
**visibilidade**: a descontinuidade sob o rótulo dividida pela de fora dele.

O teste no fim da célula fixa as duas convenções que importam: mergulho 0° é refletor deitado, e a
semblance de um refletor contínuo é ~1 mesmo quando ele mergulha.

In [3]:
class Ruler:
    FEATURES = {
        'amplitude':    ['logStd', 'kurt', 'satFrac', 'q95n'],
        'espectro':     [f'Pz{i}' for i in range(6)] + ['logPeakZ'] + [f'Px{i}' for i in range(4)],
        'estrutura':    ['cohMean', 'coh25', 'coh50', 'coh75', 'dip10', 'dip50', 'dip90'],
        'continuidade': ['lag1', 'lag2', 'lag4', 'lag8', 'lag16', 'zcr'],
        'falhas':       ['discFrac', 'lineDens', 'lineDip', 'lineLen', 'discSharp'],
        'rotulo':       ['maskFrac', 'visibility', 'maskDip', 'maskLen'],
    }
    FREQZ = np.fft.rfftfreq(TILE)

    # UM VETOR POR TILE: MÉDIA DAS 8 SEÇÕES MAIS O GRUPO DE RÓTULO NUMA SEÇÃO (A ANOTADA NO REAL, A CENTRAL NO SINTÉTICO)
    def get(self, tile, mask=None, inline=TILE // 2):
        secs = [tile[i] for i in SECTIONS] + [tile[:, :, j].T for j in SECTIONS]
        rows = pd.DataFrame([self.process(s) for s in secs])
        out  = {c: float(rows[c].mean()) if rows[c].notna().any() else np.nan for c in rows}
        return dict(out, **(self.label(tile[inline], mask) if mask is not None
                            else {k: np.nan for k in self.FEATURES['rotulo']}))

    # AS FEATURES DE IMAGEM DE UMA SEÇÃO (z, xline) EM [0,1]
    def process(self, sec):
        x   = sec.astype(np.float32)
        a   = x - np.median(x)
        sd  = float(a.std()) + 1e-8
        out = {'logStd': float(np.log(sd)), 'kurt': float((((a - a.mean()) / sd) ** 4).mean()),
               'satFrac': float(((x <= 0.002) | (x >= 0.998)).mean()),
               'q95n': float(np.percentile(a, 95) / sd)}

        vert = a - a.mean(0, keepdims=True)
        Pv   = (np.abs(np.fft.rfft(vert, axis=0)) ** 2).mean(1)
        out.update({f'Pz{i}': b for i, b in enumerate(self.bands(Pv, 6))})
        out['logPeakZ'] = float(np.log(self.FREQZ[1 + int(np.argmax(Pv[1:]))] + 1e-3))

        lat = a - a.mean(1, keepdims=True)
        out.update({f'Px{i}': b for i, b in enumerate(self.bands((np.abs(np.fft.rfft(lat, axis=1)) ** 2).mean(0), 4))})

        norm = a / sd
        gz   = cv2.Sobel(self.gauss(norm, 1.0), cv2.CV_32F, 0, 1, ksize=3)
        gx   = cv2.Sobel(self.gauss(norm, 1.0), cv2.CV_32F, 1, 0, ksize=3)
        Jzz, Jxx, Jzx = self.gauss(gz * gz, 4.0), self.gauss(gx * gx, 4.0), self.gauss(gz * gx, 4.0)
        trace = Jzz + Jxx
        coh   = np.sqrt(np.maximum((Jzz - Jxx) ** 2 + 4 * Jzx ** 2, 0)) / (trace + 1e-12)
        w     = trace / (trace.sum() + 1e-12)
        out['cohMean'] = float((coh * w).sum())
        out.update(dict(zip(['coh25', 'coh50', 'coh75'], np.percentile(coh, [25, 50, 75]))))

        theta = 0.5 * np.degrees(np.arctan2(2 * Jzx, Jxx - Jzz))
        dips  = np.where(theta > 0, theta - 90.0, theta + 90.0).ravel()
        order = np.argsort(dips)
        cum   = np.cumsum(w.ravel()[order])
        out.update({k: float(dips[order[min(np.searchsorted(cum, q), len(cum) - 1)]])
                    for k, q in zip(['dip10', 'dip50', 'dip90'], (0.1, 0.5, 0.9))})

        col = vert / (vert.std(0, keepdims=True) + 1e-8)
        out.update({f'lag{k}': float((col[:, :-k] * col[:, k:]).mean()) for k in (1, 2, 4, 8, 16)})
        sign = np.sign(vert)
        out['zcr'] = float((sign[1:] != sign[:-1]).mean())

        disc = 1.0 - self.semblance(a)
        hot  = disc > (1 - SEM_MIN)
        dips, lens = self.traces(hot)
        keep = (dips >= LINE_DIP[0]) & (dips <= LINE_DIP[1]) & (lens >= LINE_MIN)
        out['discFrac']  = float(hot.mean())
        out['lineDens']  = float(lens[keep].sum() / hot.size * 1e3)
        out['lineDip']   = float(np.median(dips[keep])) if keep.any() else np.nan
        out['lineLen']   = float(np.median(lens[keep])) if keep.any() else np.nan
        out['discSharp'] = float(np.median(disc[hot]) / (np.median(disc[~hot]) + 1e-9)) if hot.any() else np.nan
        return out

    # O RÓTULO CONTRA A IMAGEM: FRAÇÃO, VISIBILIDADE E GEOMETRIA DOS TRAÇOS ROTULADOS
    def label(self, sec, mask, minpix=20):
        out = {k: np.nan for k in self.FEATURES['rotulo']}
        out['maskFrac'] = float(mask.mean())

        if mask.sum() < minpix:
            return out

        disc = 1.0 - self.semblance(sec.astype(np.float32) - np.median(sec))
        on   = ndi.binary_dilation(mask, np.ones((3, 3), bool), iterations=2)
        off  = ~ndi.binary_dilation(mask, np.ones((3, 3), bool), iterations=8)

        if off.sum() > 100:
            out['visibility'] = float(np.median(disc[on]) / (np.median(disc[off]) + 1e-9))

        dips, lens = self.traces(mask)

        if len(dips):
            out['maskDip'], out['maskLen'] = float(np.median(dips)), float(np.median(lens))

        return out

    @staticmethod
    def gauss(img, sigma):
        return cv2.GaussianBlur(img, (0, 0), sigmaX=sigma, sigmaY=sigma, borderType=cv2.BORDER_REFLECT)

    # SEMBLANCE DE 5 TRAÇOS SEGUINDO O MERGULHO LOCAL DO TENSOR DE ESTRUTURA: ~1 NO REFLETOR CONTÍNUO, CAI NA FALHA
    @classmethod
    def semblance(cls, sec, half=2, win=4):
        smooth = cls.gauss(sec, 1.0)
        gz     = cv2.Sobel(smooth, cv2.CV_32F, 0, 1, ksize=3)
        gx     = cv2.Sobel(smooth, cv2.CV_32F, 1, 0, ksize=3)
        Jzz, Jxx, Jzx = cls.gauss(gz * gz, 4.0), cls.gauss(gx * gx, 4.0), cls.gauss(gz * gx, 4.0)
        dip    = np.clip(-np.tan(0.5 * np.arctan2(2 * Jzx, Jzz - Jxx)), -3, 3)

        zz, xx = np.mgrid[:sec.shape[0], :sec.shape[1]].astype(np.float32)
        traces = [cv2.remap(sec, xx + k, zz + dip * k, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
                  for k in range(-half, half + 1)]
        window = np.ones((2 * win + 1, 1), np.float32)
        num    = cv2.filter2D(sum(traces) ** 2, -1, window, borderType=cv2.BORDER_REFLECT)
        den    = cv2.filter2D(sum(t * t for t in traces) * len(traces), -1, window, borderType=cv2.BORDER_REFLECT)
        return num / (den + 1e-12)

    # ESPECTRO AGREGADO EM n FAIXAS LOG-ESPAÇADAS, NORMALIZADO
    @staticmethod
    def bands(power, n):
        p     = power[1:]
        edges = np.unique(np.round(np.geomspace(1, len(p), n + 1)).astype(int))
        out   = np.array([p[edges[i] - 1:edges[i + 1] - 1].sum() for i in range(n)])
        return out / (out.sum() + 1e-30)

    # MERGULHO E COMPRIMENTO DE CADA TRAÇO DE UMA MÁSCARA 2D, POR PCA
    @staticmethod
    def traces(mask, minpix=LINE_MIN):
        lbl, total = ndi.label(mask > 0, structure=np.ones((3, 3)))
        dips, lens = [], []

        for zs, xs in [np.where(lbl == i) for i in range(1, total + 1)]:
            if len(zs) < minpix:
                continue

            pts = np.stack([xs, zs]).astype(float)
            pts -= pts.mean(1, keepdims=True)
            val, vec = np.linalg.eigh(pts @ pts.T / len(zs))
            dips.append(np.degrees(np.arctan2(abs(vec[1, -1]), abs(vec[0, -1]) + 1e-12)))
            lens.append(np.sqrt(max(val[-1], 0)) * 4)

        return np.array(dips), np.array(lens)

    # CAMADAS DE MERGULHO CONHECIDO: FIXA A CONVENÇÃO DAS FEATURES DE ESTRUTURA (0° = REFLETOR DEITADO)
    def check(self, angles=(0, 30, 60), keys=('dip50', 'cohMean', 'lag8', 'discFrac', 'logPeakZ')):
        z, x  = np.mgrid[:TILE, :TILE].astype(np.float32)
        layer = lambda ang: np.sin(2 * np.pi * (z - np.tan(np.radians(ang)) * x) / 12) / 6 + 0.5
        return pd.DataFrame({f'camadas a {a}°': pd.Series(self.process(layer(a)))[list(keys)] for a in angles})


RULER = Ruler()
RULER.check().round(3)

,camadas a 0°,camadas a 30°,camadas a 60°
dip50,0.000,29.615,61.196
cohMean,1.000,0.987,0.983
lag8,1.000,-0.750,0.564
discFrac,0.000,0.000,0.073
logPeakZ,-2.443,-2.443,-2.443


# SIMILARIDADE
Cada feature vira uma nota pelo **coeficiente de energia normalizado** (Rizzo & Székely, 2016):
`H = (2A − B − C) / (2A + ε)`, com A, B e C as médias das distâncias par a par entre as duas amostras,
dentro da real e dentro da sintética. `H = 0` são distribuições iguais, então a nota da feature é
`100 × (1 − H)`. O `ε` é a resolução da feature — um quarto do IQR dela **dentro da própria região**,
com piso de 5% do IQR do bloco para features degeneradas, como `maskFrac`, que é 0 em todo tile
`calm`. A nota da região é a média aritmética dentro de cada grupo e a média geométrica ponderada
entre grupos: um grupo quebrado não é compensado por outro perfeito.

A matriz no fim da célula é a escala em uso: na diagonal, tiles reais contra a própria região; fora
dela, uma região medida contra tiles de outra.

In [4]:
class Similarity:
    WEIGHTS = {'amplitude': 0.15, 'espectro': 0.20, 'estrutura': 0.20,
               'continuidade': 0.15, 'falhas': 0.20, 'rotulo': 0.10}

    # ε DE CADA FEATURE: UM QUARTO DO IQR DELA DENTRO DA PRÓPRIA REGIÃO, COM PISO DE 5% DO IQR DO BLOCO
    def __init__(self, real, resolution=RESOLUTION, floor=0.05):
        self.real = real
        spread = lambda s: float(np.nanpercentile(s, 75) - np.nanpercentile(s, 25)) if s.notna().any() else 0.0
        block  = {f: spread(pd.concat(real.values())[f]) for names in Ruler.FEATURES.values() for f in names}
        self.eps = {region: {f: resolution * max(spread(df[f]), floor * block[f]) for f in block}
                    for region, df in real.items()}

    # A NOTA DA REGIÃO, EM %: MÉDIA GEOMÉTRICA PONDERADA DOS GRUPOS, ENTÃO UM GRUPO ZERADO DERRUBA O TODO
    def get(self, region, synth):
        groups = self.groups(region, synth) / 100
        w      = pd.Series({g: self.WEIGHTS[g] for g in groups.index})
        return 100 * float(np.exp((w / w.sum() * np.log(np.clip(groups, 1e-3, 1))).sum()))

    def groups(self, region, synth):
        return self.info(region, synth).groupby('grupo').sim.mean().dropna()

    # UMA LINHA POR FEATURE: NOTA E OS VALORES MEDIANOS DOS DOIS LADOS
    def info(self, region, synth):
        real, eps = self.real[region], self.eps[region]
        median    = lambda col: np.nanmedian(col) if col.notna().any() else np.nan
        rows = [{'grupo': group, 'feature': f, 'sim': 100 * (1 - h) if np.isfinite(h) else np.nan,
                 'real': median(real[f]), 'synth': median(synth[f])}
                for group, names in Ruler.FEATURES.items() for f in names
                for h in [self.energy(real[f], synth[f], eps[f])]]
        return pd.DataFrame(rows)

    # H DE UMA FEATURE: 0 = MESMA DISTRIBUIÇÃO, 1 = SEPARADAS
    @staticmethod
    def energy(real, synth, eps):
        r, s = np.asarray(real, float), np.asarray(synth, float)
        r, s = r[np.isfinite(r)], s[np.isfinite(s)]

        if len(r) < 5 or len(s) < 3:
            return np.nan

        A = np.abs(r[:, None] - s[None, :]).mean()
        B = np.abs(r[:, None] - r[None, :]).sum() / (len(r) * (len(r) - 1))
        C = np.abs(s[:, None] - s[None, :]).sum() / (len(s) * (len(s) - 1))
        return float(np.clip((2 * A - B - C) / (2 * A + eps), 0.0, 1.0))

    # FEATURES DE TODOS OS TILES REAIS DE UMA REGIÃO, COM A ANOTAÇÃO DO ESPECIALISTA RECORTADA EM CADA UM
    @staticmethod
    def load(region):
        db  = pd.read_csv(f'{REAL_DIR}/DataBase.csv', dtype={'patch': str})
        db  = db[db.region == region]
        ann = {p: np.asarray(Image.open(f'{PATCH_DIR}/{p}/{p}_interpretado.png').convert('L')) > 128 for p in db.patch.unique()}
        return pd.DataFrame([RULER.get(np.load(r.img_path), ann[r.patch][r.z0:r.z0 + TILE, r.x0:r.x0 + TILE], r.inline) for r in db.itertuples()])


REAL   = {r: Similarity.load(r) for r in REGIONS}
TARGET = {r: float(np.median(np.exp(REAL[r].logStd))) for r in REGIONS}
SIM    = Similarity(REAL)

print({r: f'{len(REAL[r])} tiles, sigma {TARGET[r]:.4f}' for r in REGIONS})

pd.DataFrame({a: {b: round(SIM.get(a, REAL[b].sample(N_IMAGES, random_state=2)), 1) for b in REGIONS} for a in REGIONS})

{'calm': '149 tiles, sigma 0.1930', 'faulted': '188 tiles, sigma 0.1384', 'dead': '123 tiles, sigma 0.0159'}


,calm,faulted,dead
calm,99.8,73.1,57.5
faulted,78.6,98.5,57.9
dead,58.5,63.0,99.6


# GERADOR
Um lote sintético passa pelo mesmo funil dos tiles reais: eixo `(inline, z, xline)`, saturação em
±`CLIP` e escala [0,1]. O **ganho** da região não entra na busca — é resolvido por bisseção como o
escalar que leva o σ mediano do lote ao σ mediano da região real. Contraste é o parâmetro mais fácil
de acertar e sairia dominando a nota; tirá-lo da disputa deixa a nota medir textura, estrutura e falha.

**O que a busca não toca fica no padrão da classe.** Os padrões do `SyntheticGenerator` são a
configuração calibrada do `dataset_74` e `set()` troca só as chaves passadas, então a região herda o
que já se sabe que é bom: o bloco de falha inteiro (rejeito 15–32, mergulho 55–81°, rugosidade,
arrasto, curvatura, espessura e limiar do rótulo), o `foldBaseShift`, o `shearOffset` e o
`waveletDt`. Em 2026-08-17 este projeto aprendeu que soltar os parâmetros de forma da falha contra
estatística de máscara leva a rótulo bonito sobre imagem lisa; deixá-los quietos é o que evita isso.
Medido aqui de novo em 12/09/2026: varrer `faultRoughness`, `faultRoughSigma`, `faultDecaySigma`,
`faultZoneWidth` e `faultThreshold` mexe menos de 0.6 ponto na nota do `faulted` e não fecha a
diferença de visibilidade — não vale orçamento de busca.

A tabela no fim mostra, lado a lado, o que cada região pede ao gerador — e é pela diferença entre as
colunas que se vê por que essas são as variáveis que importam.

In [5]:
class Generator:
    # O MELHOR GENOMA CONHECIDO DE CADA REGIÃO: PARTIDA DA BUSCA E CANDIDATO NA ESCOLHA FINAL
    REFERENCE = {
        'calm':    {'layerLow': 69, 'layerSpan': 54, 'thickLow': 1, 'thickSpan': 1,
                    'foldLow': 42, 'foldSpan': 35, 'foldSigmaLow': 34.95287, 'foldSigmaSpan': 31.02391,
                    'foldAmpLow': -21.71672, 'foldAmpSpan': 1.4896, 'foldDamping': 0.2,
                    'shearLow': -0.0955, 'shearSpan': 0.3763, 'faultLow': 0, 'faultSpan': 1,
                    'freqLow': 61.95206, 'freqSpan': 285.31752, 'duration': 0.095,
                    'noiseLow': 0.05, 'noiseSpan': 0.15, 'noiseSmooth': 3.0, 'noiseInline': 2.0,
                    'gainJitter': 0.26},
        'faulted': {'layerLow': 65, 'layerSpan': 24, 'thickLow': 6, 'thickSpan': 9,
                    'foldLow': 23, 'foldSpan': 13, 'foldSigmaLow': 7.82473, 'foldSigmaSpan': 50.19,
                    'foldAmpLow': -10.07881, 'foldAmpSpan': 15.6538, 'foldDamping': 3.9936,
                    'shearLow': -0.3546, 'shearSpan': 0.7, 'faultLow': 3, 'faultSpan': 3,
                    'freqLow': 38.48694, 'freqSpan': 56.08668, 'duration': 0.24,
                    'noiseLow': 0.0374, 'noiseSpan': 0.1449, 'noiseSmooth': 1.0, 'noiseInline': 1.0,
                    'gainJitter': 0.26},
        'dead':    {'layerLow': 420, 'layerSpan': 264, 'thickLow': 7, 'thickSpan': 1,
                    'foldLow': 35, 'foldSpan': 35, 'foldSigmaLow': 20.71, 'foldSigmaSpan': 5.18,
                    'foldAmpLow': 3.24, 'foldAmpSpan': 44.44, 'foldDamping': 2.9344,
                    'shearLow': -0.3078, 'shearSpan': 0.7, 'faultLow': 0, 'faultSpan': 1,
                    'freqLow': 32.98, 'freqSpan': 1.67, 'duration': 0.0257,
                    'noiseLow': 0.12, 'noiseSpan': 0.28, 'noiseSmooth': 2.0, 'noiseInline': 1.0,
                    'gainJitter': 0.35},
    }

    # UM LOTE DA REGIÃO: TILES EM [0,1] COM A MÁSCARA, AS FEATURES E O GANHO AJUSTADO
    def get(self, region, genome, seed=SEED, n=N_IMAGES):
        with ProcessPoolExecutor(NJOBS, mp_context=mp.get_context('fork')) as pool:
            raws = list(pool.map(self.tile, [(self.options(genome), seed + i) for i in range(n)]))

        jit   = self.jitter(genome['gainJitter'], seed, n)
        gain  = self.gain(raws, jit, TARGET[region])
        tiles = [(self.format(img, gain, j), msk) for (img, msk), j in zip(raws, jit)]
        feats = pd.DataFrame([RULER.get(img, msk[TILE // 2]) for img, msk in tiles])
        return tiles, feats, gain

    # AS OPÇÕES DA REGIÃO: SÓ O QUE A BUSCA MEXE, CADA PAR (low, span) VIRANDO O INTERVALO (low, low + span)
    @staticmethod
    def options(genome):
        pair = lambda lo, span: (genome[lo], genome[lo] + genome[span])
        return dict(layerRange=pair('layerLow', 'layerSpan'), layerThickness=pair('thickLow', 'thickSpan'),
                    foldCount=pair('foldLow', 'foldSpan'), foldSigma=pair('foldSigmaLow', 'foldSigmaSpan'),
                    foldAmplitude=pair('foldAmpLow', 'foldAmpSpan'), foldDamping=genome['foldDamping'],
                    shearGradient=pair('shearLow', 'shearSpan'), faultCount=pair('faultLow', 'faultSpan'),
                    waveletFreq=pair('freqLow', 'freqSpan'), waveletDuration=genome['duration'],
                    noiseLevel=pair('noiseLow', 'noiseSpan'),
                    noiseSigma=(genome['noiseInline'], genome['noiseSmooth'], genome['noiseSmooth']))

    # UM TILE CRU, JÁ NO EIXO (inline, z, xline) DO SLAB REAL
    @staticmethod
    def tile(args):
        options, seed = args
        np.random.seed(seed)
        gen = SyntheticGenerator(shape=(TILE,) * 3)
        gen.set(options)
        img, msk = gen.get()
        return np.transpose(img, (0, 2, 1)).astype(np.float32), np.transpose(msk, (0, 2, 1)).astype(bool)

    # O JITTER DE CONTRASTE DO LOTE, COM MEDIANA 1: SEM ISSO O GANHO ABSORVE O VIÉS DO SORTEIO E SÓ
    # VALE PARA UM LOTE DAQUELE TAMANHO — EM 18 TILES A MEDIANA DO SORTEIO CHEGA A 1.17
    @staticmethod
    def jitter(amount, seed, n):
        draw = np.exp(amount * np.clip(np.random.RandomState(seed).normal(0, 1, n), -2, 2))
        return draw / np.median(draw)

    # A SATURAÇÃO SIMÉTRICA DO SLAB REAL, DEPOIS DO GANHO DA REGIÃO E DO JITTER DO TILE
    @staticmethod
    def format(img, gain, jit):
        return np.clip(np.clip(img * gain * jit, -CLIP, CLIP) / (2 * CLIP) + 0.5, 0, 1).astype(np.float32)

    # GANHO DA REGIÃO: O ESCALAR QUE LEVA O σ MEDIANO PÓS-FORMAT AO σ MEDIANO DOS TILES REAIS, POR BISSEÇÃO
    def gain(self, raws, jitter, target, iters=40):
        sample = np.stack([img.ravel()[::37] for img, _ in raws])
        lo, hi = 1e-4, 10.0

        for _ in range(iters):
            mid   = np.sqrt(lo * hi)
            sigma = np.median([self.format(s, mid, j).std() for s, j in zip(sample, jitter)])
            lo, hi = (mid, hi) if sigma < target else (lo, mid)

        return float(np.sqrt(lo * hi))


GEN = Generator()
pd.DataFrame({r: {k: str(np.round(v, 3) if isinstance(v, tuple) else round(v, 4)) for k, v in GEN.options(GEN.REFERENCE[r]).items()} for r in REGIONS})

,calm,faulted,dead
layerRange,[ 69 123],[65 89],[420 684]
layerThickness,[1 2],[ 6 15],[7 8]
foldCount,[42 77],[23 36],[35 70]
foldSigma,[34.953 65.977],[ 7.825 58.015],[20.71 25.89]
foldAmplitude,[-21.717 -20.227],[-10.079 5.575],[ 3.24 47.68]
foldDamping,0.2,3.9936,2.9344
shearGradient,[-0.096 0.281],[-0.355 0.345],[-0.308 0.392]
faultCount,[0 1],[3 6],[0 1]
waveletFreq,[ 61.952 347.27 ],[38.487 94.574],[32.98 34.65]
waveletDuration,0.095,0.24,0.0257


# ESPAÇO DE BUSCA
**23 variáveis, só as que separam uma região da outra.** O que muda de região para região é a
estratigrafia (`layerRange`, `layerThickness`), o dobramento (`foldCount`, `foldSigma`,
`foldAmplitude`, `foldDamping`, `shearGradient`), a wavelet (`waveletFreq`, `waveletDuration`), o
ruído (`noiseLevel` e o grão dele, `noiseSmooth` e `noiseInline`) e a quantidade de falha
(`faultCount`) — mais o `gainJitter`, que é a variação de contraste entre tiles. Cada intervalo entra
como um par `low` + `span`, então ele sai sempre válido e a busca não precisa de restrição.

O grão do ruído (`noiseSigma = (noiseInline, noiseSmooth, noiseSmooth)`) entrou por medição: a zona
morta real tem coerência 0.64 **com** correlação lateral 0.79, o que só acontece com ruído liso e
isotrópico no plano da seção. Com o grão achatado que era fixo na classe (σz 0.5 contra σx 1.0) o
ruído vira textura deitada e a coerência não desce. Soltando o grão, o `dead` sai de 74 para 83 e o
`calm` de 85 para 88 — foi o que motivou expor `noiseSigma` no `SyntheticGenerator`, com o padrão da
classe idêntico ao comportamento antigo (conferido bit a bit).

A faixa de `freqLow`, `freqSpan` e `duration` está na escala do `waveletDt = 0.0012` do padrão: a
wavelet depende de `freq × dt` e o kernel de `duration / dt`, então os valores do `Generator2`, que
usava `dt = 0.002`, entram na `REFERENCE` multiplicados por 1.667 e por 0.6 — conferido, nessa
escala as features saem idênticas.

**A busca parte de onde já se chegou.** Mesmo com 22 variáveis a caixa larga é espaço demais para o
orçamento de uma chamada: partindo de um ponto aleatório, 32 avaliações chegaram a 72% no `calm`,
abaixo dos 84% que o genoma do `Generator2.ipynb` já dá. Então a `Generator.REFERENCE` vale duas coisas: a
caixa da região é uma **vizinhança** dele (`SPREAD` de meia-largura, mínimo ±1 nas variáveis
inteiras) e ele disputa a escolha final. O `SPREAD` é por região porque o quanto se confia na
referência é por região; `None` varre a caixa inteira, que é o modo de uma campanha longa.

**A decisão final é em sementes novas.** A busca otimiza sempre nas mesmas 18 sementes e acaba
aprendendo os defeitos daquele conjunto: na primeira rodada o `calm` marcou 88.0% nas sementes da
busca e 82.9% fora delas, contra um ruído de ±1.3. Por isso `export` mede as três candidatas — o
achado da busca, a referência e o que já estava no `regions.json` — num lote de sementes novas e
grava a que generaliza melhor.

In [6]:
class Search:
    SPACE = {
        'layerLow':      {'bounds': (10, 500),   'type': 'int'},   # camadas na coluna
        'layerSpan':     {'bounds': (1, 300),    'type': 'int'},
        'thickLow':      {'bounds': (1, 12),     'type': 'int'},   # espessura da camada, em voxels
        'thickSpan':     {'bounds': (1, 12),     'type': 'int'},
        'foldLow':       {'bounds': (2, 60),     'type': 'int'},   # nº de dobras
        'foldSpan':      {'bounds': (1, 40),     'type': 'int'},
        'foldSigmaLow':  {'bounds': (3.0, 80.0)},                  # largura da dobra
        'foldSigmaSpan': {'bounds': (1.0, 70.0)},
        'foldAmpLow':    {'bounds': (-50.0, 40.0)},                # altura da dobra
        'foldAmpSpan':   {'bounds': (0.0, 60.0)},
        'foldDamping':   {'bounds': (0.2, 6.0)},                   # perda da dobra com a profundidade
        'shearLow':      {'bounds': (-0.5, 0.2)},                  # mergulho regional
        'shearSpan':     {'bounds': (0.0, 0.7)},
        'faultLow':      {'bounds': (0, 3),      'type': 'int'},   # nº de falhas por tile
        'faultSpan':     {'bounds': (1, 3),      'type': 'int'},
        'freqLow':       {'bounds': (25.0, 333.0)},                # frequência da wavelet, na escala do dt=0.0012
        'freqSpan':      {'bounds': (0.0, 300.0)},
        'duration':      {'bounds': (0.018, 0.240)},               # duração da wavelet
        'noiseLow':      {'bounds': (0.0, 0.40)},                  # ruído: o que quebra a coerência do refletor
        'noiseSpan':     {'bounds': (0.0, 0.60)},
        'noiseSmooth':   {'bounds': (0.4, 5.0)},                   # grão do ruído na seção (z, xline)
        'noiseInline':   {'bounds': (0.4, 5.0)},                   # grão do ruído ao longo das inlines
        'gainJitter':    {'bounds': (0.0, 0.6)},                   # variação de contraste entre tiles
    }

    def __init__(self, spread=SPREAD, population=POPULATION, generations=GENERATIONS, patience=PATIENCE):
        self.spread      = spread
        self.population  = population
        self.generations = generations
        self.patience    = patience

    # A CAMPANHA DA REGIÃO, RETOMÁVEL. A PASTA LEVA A LARGURA DA CAIXA: O ESTADO DO CMA-ES ESTÁ EM
    # UNIDADES REAIS, ENTÃO RETOMAR COM OUTRA CAIXA PARTIRIA DE UMA MÉDIA FORA DELA
    def update(self, region):
        start      = time.time()
        spread     = self.spread[region]
        self.model = NatureSelector('genetic', {
            'objective': self.objective(region), 'variables': self.space(region), 'maximize': True,
            'population': self.population, 'generations': self.generations,
            'patience': self.patience, 'seed': SEED, 'verbose': True,
        }, memory=f'{MEMORY_DIR}/{region}_{"full" if spread is None else round(100 * spread)}')

        self.best, self.score = self.model.update()
        print(f'\n[{region}] busca: {self.score:.1f}% nas sementes da busca  '
              f'({self.model.optimizer.memory.runs} chamada(s), {(time.time() - start) / 60:.1f} min)')
        return self.export(region)

    # A CAIXA DA REGIÃO: VIZINHANÇA DA REFERÊNCIA, COM PELO MENOS ±1 NAS VARIÁVEIS INTEIRAS
    def space(self, region):
        spread = self.spread[region]

        if spread is None:
            return self.SPACE

        box = {}

        for name, spec in self.SPACE.items():
            lo, hi = spec['bounds']
            half   = max(spread * (hi - lo), 1 if spec.get('type') == 'int' else 0)
            box[name] = {**spec, 'bounds': (max(lo, GEN.REFERENCE[region][name] - half),
                                            min(hi, GEN.REFERENCE[region][name] + half))}

        return box

    # O OBJETIVO: A NOTA DO LOTE DA REGIÃO CONTRA OS TILES REAIS; CONFIGURAÇÃO QUE NEM GERA VALE 0
    def objective(self, region):
        def score(genome):
            try:
                return SIM.get(region, GEN.get(region, genome)[1])
            except Exception as error:
                print(f'[{region}] configuração descartada: {error}')
                return 0.0

        return score

    # ESCOLHE EM SEMENTES NOVAS ENTRE A BUSCA, A REFERÊNCIA E O QUE JÁ ESTAVA GRAVADO, E GRAVA A VENCEDORA
    def export(self, region, path=REGIONS_PATH):
        data     = json.load(open(path)) if os.path.exists(path) else {}
        saved    = data.get('regions', {}).get(region, {})
        trials   = {'busca': self.best, 'referencia': GEN.REFERENCE[region]}
        searched = {'busca': round(float(self.score), 2)}

        if saved.get('genome', {}).keys() == GEN.REFERENCE[region].keys():
            trials['gravado']   = saved['genome']
            searched['gravado'] = saved.get('scoreSearch')

        runs   = {name: GEN.get(region, genome, CHECK_SEED) for name, genome in trials.items()}
        checks = {name: SIM.get(region, feats) for name, (_, feats, _) in runs.items()}
        winner = max(checks, key=checks.get)
        _, feats, gain = runs[winner]

        data.update({'tile': TILE, 'nImages': N_IMAGES, 'clip': CLIP, 'seed': SEED, 'updated': datetime.now().strftime('%d/%m/%y %H:%M:%S')})
        data.setdefault('regions', {})[region] = {
            'score': round(checks[winner], 2), 'scoreSearch': searched.get(winner),
            'source': saved.get('source', winner) if winner == 'gravado' else winner,
            'gain': round(gain, 6), 'groups': SIM.groups(region, feats).round(1).to_dict(),
            'genome': {k: int(v) if isinstance(v, (int, np.integer)) else round(float(v), 5)
                       for k, v in trials[winner].items()},
            'params': {k: list(v) if isinstance(v, tuple) else v for k, v in GEN.options(trials[winner]).items()},
        }

        json.dump(data, open(path, 'w'), ensure_ascii=False, indent=4)
        print(f'[{region}] em sementes novas: ' + '  '.join(f'{k} {v:.1f}%' for k, v in checks.items()) + f'  -> fica a {winner} (ganho {gain:.4f}) -> {path}')
        return data['regions'][region]

    # TILES REAIS COM A ANOTAÇÃO CONTRA SINTÉTICOS COM A MÁSCARA, DO LOTE DA CONFIGURAÇÃO GRAVADA
    def show(self, region, n=3, path=REGIONS_PATH):
        saved = json.load(open(path))['regions'][region]
        tiles, feats, gain = GEN.get(region, saved['genome'], CHECK_SEED)
        db  = pd.read_csv(f'{REAL_DIR}/DataBase.csv', dtype={'patch': str})
        db  = db[db.region == region].sample(n, random_state=7)
        ann = {p: np.asarray(Image.open(f'{PATCH_DIR}/{p}/{p}_interpretado.png').convert('L')) > 128 for p in db.patch.unique()}
        fig, axes = plt.subplots(2, n, figsize=(4.6 * n, 9.6))

        for ax, row in zip(axes[0], db.itertuples()):
            sec    = np.load(row.img_path)[row.inline]
            zf, xf = np.nonzero(ann[row.patch][row.z0:row.z0 + TILE, row.x0:row.x0 + TILE])
            ax.imshow(sec, cmap='gray', vmin=0, vmax=1)
            ax.scatter(xf, zf, s=1, c='lime')
            ax.set_title(f'REAL {row.id}\nσ={sec.std():.3f}', fontsize=9)

        for ax, (img, msk) in zip(axes[1], tiles):
            sec, mask = img[TILE // 2], msk[TILE // 2]
            zf, xf    = np.nonzero(mask)
            ax.imshow(sec, cmap='gray', vmin=0, vmax=1)
            ax.scatter(xf, zf, s=1, c='red')
            ax.set_title(f'SINTÉTICO (falha={mask.mean() * 100:.1f}%)\nσ={sec.std():.3f}', fontsize=9)

        for ax in axes.ravel():
            ax.axis('off')

        fig.suptitle(f'REGIÃO {region.upper()} — nota {SIM.get(region, feats):.1f}% em {len(feats)} tiles '
                     f'(ganho {gain:.4f})', fontsize=13)
        plt.tight_layout(rect=(0, 0, 1, 0.97))
        plt.show()
        return SIM.info(region, feats).dropna(subset=['sim']).sort_values('sim').head(8).round(3)


SEARCH = Search()

# REGIÃO CALM
Pacote raso: refletores finos, contínuos e quase deitados, sem falha anotada por perto. Rodar a
célula de novo **estende** a campanha a partir da memória.

In [7]:
SEARCH.update('calm')

genetic:   3%|▎         | 36/1200 [26:46<13:10:10, 40.73s/ev, best=86.0274, pop=12]

KeyboardInterrupt: 

In [ ]:
SEARCH.show('calm')

# REGIÃO FAULTED
Intervalo produtivo: refletores de frequência mais baixa cortados por falhas de mergulho alto — é a
região em que os grupos `falhas` e `rotulo` mandam na nota.

In [ ]:
SEARCH.update('faulted')

In [ ]:
SEARCH.show('faulted')

# REGIÃO DEAD
Zona morta: amplitude ~8x menor, nenhum refletor coerente e o crosshatch de migração. É a mais difícil
das três — a `SyntheticGenerator` monta refletividade 1D e convolve em z, então a coerência não desce
de ~0.97 e o real fica em 0.68; a nota aqui nasce limitada pelo gerador, não pela métrica.

In [ ]:
SEARCH.update('dead')

In [ ]:
SEARCH.show('dead')

# RESULTADO
`regions.json` guarda, por região, o genoma, as opções prontas para o `SyntheticGenerator`, o ganho,
de onde veio a configuração e as duas notas — `score` é a que vale, medida em sementes que a busca
não viu. Para gerar um dataset com elas:

```python
cfg = json.load(open('regions.json'))['regions']
gen = SyntheticGenerator(shape=(128, 128, 128))
gen.dataset({'directory': 'files', 'regions': {r: {'n_images': 200, 'output': r, 'params': cfg[r]['params']}
                                               for r in cfg}})
```

Os tiles saem z-scorados; para cair na escala do bloco real, passe cada um pelo `formatTile` com o
`gain` da região e o jitter do `genome.gainJitter` antes do `Format.ipynb`.

In [ ]:
cfg = json.load(open(REGIONS_PATH))
print(f"atualizado em {cfg['updated']}  |  {cfg['nImages']} tiles por lote")
pd.DataFrame({r: {'nota': v['score'], 'na busca': v['scoreSearch'], 'origem': v['source'], 'ganho': v['gain'], **v['groups']} for r, v in cfg['regions'].items()}).T